# Coordinate Compression

A small but high-leverage preprocessing trick: when values are **sparse or huge** but only their **relative order** matters, remap them to dense ranks `0 .. k-1`. Then you can index a size-k array (or a Fenwick tree) *by value* even when the values span billions — without a billion-slot array.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **The compression**
3. **The payoff** — indexing by value
4. **When to use & cheat-sheet**

## 1. The signal — only the order matters

Reach for coordinate compression when:

- the values are **large or sparse** (timestamps, prices, coordinates up to 10⁹) but there are only **k distinct** ones, and
- the algorithm needs to **index an array or a Fenwick / segment tree by value** — count-smaller-to-the-right, range queries over value space, sweep-line events — where only the **relative order** of values matters, not their magnitudes.

The fix: replace each value by its **rank** — its position in the sorted list of distinct values. Ranks are `0 .. k-1`: contiguous, order-preserving, and small enough to index a plain array.

## 2. The compression

Sort the **distinct** values, then map each to its index. That's the whole trick — `sorted(set(...))` plus a dict, O(n log n) once. Because the mapping is monotonic, **comparisons survive**: `a < b` iff `rank[a] < rank[b]`.

In [1]:
def compress(values):
    order = sorted(set(values))                  # the distinct values, in order
    rank = {v: i for i, v in enumerate(order)}   # value -> its dense rank (0 .. k-1)
    return [rank[v] for v in values], order, rank

values = [100, 3, 100, 999, 3, -5]
compressed, order, rank = compress(values)
print("values     :", values)
print("order       :", order)        # [-5, 3, 100, 999]
print("compressed  :", compressed)   # [2, 1, 2, 3, 1, 0]

# order is preserved -> compressed values compare the same way:
print("3 < 100  <->  rank[3] < rank[100]:", (3 < 100) == (rank[3] < rank[100]))


values     : [100, 3, 100, 999, 3, -5]
order       : [-5, 3, 100, 999]
compressed  : [2, 1, 2, 3, 1, 0]
3 < 100  <->  rank[3] < rank[100]: True


Keep `order` around — it's the **inverse** map (`order[r]` recovers the original value from a rank), which you need to translate answers back out of rank-space.

## 3. The payoff — indexing by value, and ranking new values

Two things compression buys you:

**Index a tiny array by value.** A counts / Fenwick array now needs only **k** slots (one per distinct value), not `max_value + 1` — so value-indexed structures work even when values reach 10⁹. And `bisect` ranks *any* value, including ones you didn't precompute:

In [2]:
# bucket counts indexed by rank: a size-k array, not a size-1000 one
counts = [0] * len(order)
for v in values:
    counts[rank[v]] += 1
print("counts by rank:", counts, "(size", len(order), "- not 1000)")

# rank ANY value (even one not in the set) in O(log k) with bisect:
import bisect
def rank_of(x):
    return bisect.bisect_left(order, x)

print("rank_of(100):", rank_of(100), "| rank_of(50):", rank_of(50), "(where 50 would sit)")


counts by rank: [1, 2, 2, 1] (size 4 - not 1000)
rank_of(100): 2 | rank_of(50): 2 (where 50 would sit)


`bisect_left` against the sorted `order` ranks values for **online** queries or for placing query points relative to the compressed coordinates — the same `bisect` from the searching notebook, now used as a value→rank function. Compressed ranks are exactly what you feed a **Fenwick tree** (the Fenwick & segment trees notebook) to answer "how many values seen so far are < x" over a huge value range.

## 4. When to use & cheat-sheet

| You have… | Compression lets you… |
|---|---|
| values up to 10⁹ but only k distinct | index a size-k array / Fenwick by value |
| "count of smaller elements to the right" | run a BIT over compressed ranks |
| interval / sweep-line events on a huge axis | relabel coordinates to `0 .. k-1` |
| "only relative order matters" | drop magnitudes entirely |

**Python toolkit:** `sorted(set(values))` builds the order list; `{v: i for i, v in enumerate(order)}` is the value→rank map; `bisect.bisect_left(order, x)` ranks any value (including unseen ones) in O(log k); index `order[r]` to map a rank back.

**Skip it when:** the value range is already small (just use the values as indices), or you never index by value (a `dict` keyed by the raw value is simpler than ranks).

| Step | How | Cost |
|---|---|:---:|
| Build the rank map | `sorted(set(values))` + dict | O(n log n) |
| Compress a known value | `rank[v]` | O(1) |
| Rank an arbitrary value | `bisect_left(order, v)` | O(log k) |
| Decompress a rank | `order[r]` | O(1) |